# Coffee17 WRC-HBP V1 — WR-HBP + CLAHE Contrast Residual

Matched development experiment:

`WR-HBP`

vs

`WRC-HBP = WR-HBP + zero-gated C0/CLAHE-derived luminance contrast residual`

Both arms are retrained on **5 folds, seed42**. The complete shared WR-HBP
state is verified identical before training; the new contrast gate starts at
zero.

Kaggle Input: **Coffee Green Bean with 17 Defects Original** only.

Outer test is not accessed.


In [ ]:
NOTEBOOK_BUILD = "WRC-HBP-V1"
SCIENTIFIC_CODE_COMMIT = "762e0af63b911b9fa2c695b16214e37aa085a723"
print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)

import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-wrc-hbp-code"
PROJECT = WORK / "coffee17-wavelet-clahe-residual-hbp-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# Optional resume from a previous run of this exact project.
prior_projects = sorted(
    p for p in INPUT.rglob("coffee17-wavelet-clahe-residual-hbp-project")
    if p.is_dir()
)
if len(prior_projects) > 1:
    raise RuntimeError("Lebih dari satu prior WRC-HBP project ditemukan.")
if prior_projects:
    print("MERGE PRIOR WRC-HBP OUTPUT:", prior_projects[0])
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    shutil.copytree(prior_projects[0], PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Exact scientific-code checkout.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)

run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run([
    "git", "-C", REPO, "checkout", "--quiet", "--detach",
    SCIENTIFIC_CODE_COMMIT
])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)
from bilinear_lmmd.core.config import load_config
from bilinear_lmmd.engine.wavelet_clahe_residual_hbp import (
    preflight_matched_initialization,
)

CONFIG = REPO / "configs/wavelet_clahe_residual_hbp/WRC_HBP_V1.yaml"
OUT = PROJECT / "experiments/coffee17-wavelet-clahe-residual-hbp-v1"
OUT.mkdir(parents=True, exist_ok=True)

# Static matched initialization preflight.
cfg = load_config(CONFIG)
preflight = preflight_matched_initialization(cfg)
print("\n=== INITIALIZATION PREFLIGHT ===")
print(json.dumps(preflight, indent=2))

# ------------------------------------------------------------
# 2. Reconstruct exact Coffee17 development folds.
# ------------------------------------------------------------
print("\n=== 1/4 RECONSTRUCT COFFEE17 ===")

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_wrc_hbp_original.zip"
PROV_LOCAL = WORK / "coffee17_wrc_hbp_provenance"
CANONICAL = WORK / "coffee17_wrc_hbp_original_v1"
FOLDS_LOCAL = WORK / "coffee17_wrc_hbp_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV_LOCAL,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

# ------------------------------------------------------------
# 3. Five matched folds.
# ------------------------------------------------------------
print("\n=== 2/4 MATCHED WR-HBP VS WRC-HBP ===")

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_wrc_hbp_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL / "clean_manifest.json",
        FOLDS_LOCAL / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_wavelet_clahe_residual_hbp_matched",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"WRC_HBP_fold{fold}.log")

    result = json.loads(
        (OUT / f"fold_{fold}" / "seed42" / "pair_result.json").read_text(
            encoding="utf-8"
        )
    )
    d = result["DELTA_WRC_MINUS_WR"]
    print(
        f"FOLD {fold} | "
        f"WR Macro={result['WR_HBP']['macro_f1']:.4f} | "
        f"WRC Macro={result['WRC_HBP']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f} | "
        f"WaveletGate={result['wavelet_gate_at_best']['WRC_HBP']:+.5f} | "
        f"ContrastGate={result['contrast_gate_at_best']:+.5f}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Paired summary.
# ------------------------------------------------------------
print("\n=== 3/4 PAIRED SUMMARY ===")
SUMMARY = PROJECT / "analysis" / "wavelet_clahe_residual_hbp_summary.json"

run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_wavelet_clahe_residual_hbp_summary",
    "--output-root", OUT,
    "--config", CONFIG,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
agg = summary["aggregate"]
delta = agg["DELTA_WRC_MINUS_WR"]

print("\n=== KEY RESULT ===")
print("WR-HBP Macro-F1  :", f"{agg['WR_HBP']['macro_f1']['mean']:.4%}")
print("WRC-HBP Macro-F1 :", f"{agg['WRC_HBP']['macro_f1']['mean']:.4%}")
print("Delta Macro-F1   :", f"{delta['macro_f1']['mean']:+.4%}")
print("WR-HBP Hard-F1   :", f"{agg['WR_HBP']['hard_class_f1']['mean']:.4%}")
print("WRC-HBP Hard-F1  :", f"{agg['WRC_HBP']['hard_class_f1']['mean']:.4%}")
print("Delta Hard-F1    :", f"{delta['hard_class_f1']['mean']:+.4%}")
print("WR-HBP Worst-F1  :", f"{agg['WR_HBP']['worst_class_f1']['mean']:.4%}")
print("WRC-HBP Worst-F1 :", f"{agg['WRC_HBP']['worst_class_f1']['mean']:.4%}")
print("Delta Worst-F1   :", f"{delta['worst_class_f1']['mean']:+.4%}")
print("Positive Macro folds:", summary["positive_delta_folds"]["macro_f1"], "/ 5")
print("Rescue / damage      :",
      summary["paired_prediction_outcomes"]["rescue"], "/",
      summary["paired_prediction_outcomes"]["damage"])
print("Mean contrast gate   :",
      f"{summary['contrast_gate_at_best']['mean']:+.6f}")
print("Parameter overhead   :",
      summary["parameter_counts"]["candidate_overhead"],
      f"({summary['parameter_counts']['candidate_overhead_percent']:.3f}%)")
print("SCREENING GATE       :", summary["screening_gate"]["decision"])

# ------------------------------------------------------------
# 5. Compact analysis package.
# ------------------------------------------------------------
print("\n=== 4/4 BUILD ANALYSIS PACKAGE ===")

PKG = WORK / "wavelet-clahe-residual-hbp-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copytree(PROJECT / "analysis", PKG / "analysis")

for fold in range(1, 6):
    pair_src = OUT / f"fold_{fold}" / "seed42"
    pair_dst = PKG / f"fold_{fold}"
    pair_dst.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pair_src / "pair_result.json", pair_dst / "pair_result.json")

    for arm in ("WR_HBP", "WRC_HBP"):
        src = pair_src / arm
        dst = pair_dst / arm
        dst.mkdir(parents=True, exist_ok=True)
        for name in ("run_contract.json", "history.json", "run_config.yaml"):
            p = src / name
            if p.is_file():
                shutil.copy2(p, dst / name)
        if (src / "validation").is_dir():
            shutil.copytree(src / "validation", dst / "validation")

shutil.copy2(CONFIG, PKG / "WRC_HBP_V1.yaml")
protocol = REPO / "docs/protocols/WAVELET_CLAHE_RESIDUAL_HBP_V1.md"
if protocol.is_file():
    shutil.copy2(protocol, PKG / "WAVELET_CLAHE_RESIDUAL_HBP_V1.md")

zip_path = Path(
    shutil.make_archive(
        str(WORK / "wavelet-clahe-residual-hbp-analysis-package"),
        "zip",
        root_dir=PKG,
    )
)

print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("PROJECT:", PROJECT)
print("Outer test was NOT accessed.")

for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
